# 02 — silver_orders
**Entrée :** `bronze.orders` + `bronze.customers` (pour `customer_unique_id`)
**Sortie :** `silver.silver_orders` — **99 441 lignes** (autant que Bronze)
**Ce que fait ce notebook :** convertit les 5 dates en TIMESTAMP (`TRY_CAST`), met le statut en minuscules,
ajoute 8 indicateurs `flag_…` (anomalies R01 à R09). **Aucune ligne n'est supprimée.**

In [0]:
CATALOG = "workspace"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
# Les tables Bronze sont supposées s'appeler : orders, customers, order_items, order_payments,
# order_reviews, products, sellers, product_category_name_translation.
# Si les tiennes ont d'autres noms, change-les dans les requêtes (Ctrl+F).

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")


def scalar(sql):
    """Exécute une requête qui renvoie une seule valeur."""
    return spark.sql(sql).collect()[0][0]


def check(label, obtenu, attendu):
    """Compare ce qu'on obtient à ce qu'on attend (valeurs de ton document de travail)."""
    ok = obtenu == attendu
    print(("✅ OK     " if ok else "❌ ÉCART  ") + f"{label} : obtenu = {obtenu} | attendu = {attendu}")

## 1. Construire la table

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {SILVER}.silver_orders AS
WITH o AS (
    SELECT
        order_id,
        customer_id,
        LOWER(TRIM(order_status)) AS order_status,
        TRY_CAST(NULLIF(TRIM(order_purchase_timestamp), '')      AS TIMESTAMP) AS order_purchase_ts,
        TRY_CAST(NULLIF(TRIM(order_approved_at), '')             AS TIMESTAMP) AS order_approved_ts,
        TRY_CAST(NULLIF(TRIM(order_delivered_carrier_date), '')  AS TIMESTAMP) AS order_carrier_ts,
        TRY_CAST(NULLIF(TRIM(order_delivered_customer_date), '') AS TIMESTAMP) AS order_delivered_ts,
        TRY_CAST(NULLIF(TRIM(order_estimated_delivery_date), '') AS TIMESTAMP) AS order_estimated_ts,
        _loaded_at
    FROM {BRONZE}.orders
)
SELECT
    o.order_id,
    o.customer_id,
    c.customer_unique_id,
    o.order_status,
    o.order_purchase_ts, o.order_approved_ts, o.order_carrier_ts, o.order_delivered_ts, o.order_estimated_ts,
    -- Indicateurs (TRUE = anomalie). COALESCE : un NULL ne doit jamais donner un indicateur NULL.
    COALESCE(o.order_status = 'delivered' AND o.order_delivered_ts IS NULL, FALSE) AS flag_r01_delivered_no_delivery_date,
    COALESCE(o.order_status = 'delivered' AND o.order_approved_ts  IS NULL, FALSE) AS flag_r02_delivered_no_approval,
    COALESCE(o.order_status = 'delivered' AND o.order_carrier_ts   IS NULL, FALSE) AS flag_r03_delivered_no_carrier,
    COALESCE(o.order_status = 'canceled'  AND o.order_delivered_ts IS NOT NULL, FALSE) AS flag_r04_canceled_with_delivery,
    COALESCE(o.order_carrier_ts   < o.order_purchase_ts, FALSE) AS flag_r05_carrier_before_purchase,
    COALESCE(o.order_carrier_ts   < o.order_approved_ts, FALSE) AS flag_r06_carrier_before_approval,
    COALESCE(o.order_delivered_ts < o.order_approved_ts, FALSE) AS flag_r08_delivery_before_approval,
    COALESCE(o.order_delivered_ts < o.order_carrier_ts,  FALSE) AS flag_r09_delivery_before_carrier,
    o._loaded_at AS _bronze_loaded_at,
    current_timestamp() AS _silver_loaded_at
FROM o
LEFT JOIN {BRONZE}.customers c ON o.customer_id = c.customer_id
""")
print("silver_orders créée")

silver_orders créée


## 2. Contrôles : lignes et personnes

In [0]:
t = f"{SILVER}.silver_orders"
check("lignes", scalar(f"SELECT COUNT(*) FROM {t}"), 99441)
check("personnes (customer_unique_id)", scalar(f"SELECT COUNT(DISTINCT customer_unique_id) FROM {t}"), 96096)
check("customer_unique_id NULL", scalar(f"SELECT COUNT(*) FROM {t} WHERE customer_unique_id IS NULL"), 0)
check("order_id doublons", scalar(f"SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM {t}"), 0)

✅ OK     lignes : obtenu = 99441 | attendu = 99441
✅ OK     personnes (customer_unique_id) : obtenu = 96096 | attendu = 96096
✅ OK     customer_unique_id NULL : obtenu = 0 | attendu = 0
✅ OK     order_id doublons : obtenu = 0 | attendu = 0


## 3. Contrôles : les indicateurs (valeurs trouvées à l'exploration)

In [0]:
attendus = {
    "flag_r01_delivered_no_delivery_date": 8,
    "flag_r02_delivered_no_approval": 14,
    "flag_r03_delivered_no_carrier": 2,
    "flag_r04_canceled_with_delivery": 6,
    "flag_r05_carrier_before_purchase": 166,
    "flag_r06_carrier_before_approval": 1359,
    "flag_r08_delivery_before_approval": 61,
    "flag_r09_delivery_before_carrier": 23,
}
for flag, attendu in attendus.items():
    check(flag, scalar(f"SELECT COUNT(*) FROM {t} WHERE {flag}"), attendu)

✅ OK     flag_r01_delivered_no_delivery_date : obtenu = 8 | attendu = 8
✅ OK     flag_r02_delivered_no_approval : obtenu = 14 | attendu = 14
✅ OK     flag_r03_delivered_no_carrier : obtenu = 2 | attendu = 2
✅ OK     flag_r04_canceled_with_delivery : obtenu = 6 | attendu = 6
✅ OK     flag_r05_carrier_before_purchase : obtenu = 166 | attendu = 166
✅ OK     flag_r06_carrier_before_approval : obtenu = 1359 | attendu = 1359
✅ OK     flag_r08_delivery_before_approval : obtenu = 61 | attendu = 61
✅ OK     flag_r09_delivery_before_carrier : obtenu = 23 | attendu = 23


## 4. Contrôle : aucune conversion de date n'a échoué
Une conversion échoue quand Bronze contient une valeur non vide mais que Silver est NULL.

In [0]:
paires = {
    "order_purchase_timestamp": "order_purchase_ts",
    "order_approved_at": "order_approved_ts",
    "order_delivered_carrier_date": "order_carrier_ts",
    "order_delivered_customer_date": "order_delivered_ts",
    "order_estimated_delivery_date": "order_estimated_ts",
}
for col_bronze, col_silver in paires.items():
    n = scalar(f"""
        SELECT COUNT(*) FROM {BRONZE}.orders b JOIN {t} s ON b.order_id = s.order_id
        WHERE NULLIF(TRIM(b.{col_bronze}), '') IS NOT NULL AND s.{col_silver} IS NULL""")
    check(f"conversions échouées {col_silver}", n, 0)

✅ OK     conversions échouées order_purchase_ts : obtenu = 0 | attendu = 0
✅ OK     conversions échouées order_approved_ts : obtenu = 0 | attendu = 0
✅ OK     conversions échouées order_carrier_ts : obtenu = 0 | attendu = 0
✅ OK     conversions échouées order_delivered_ts : obtenu = 0 | attendu = 0
✅ OK     conversions échouées order_estimated_ts : obtenu = 0 | attendu = 0


## 5. Tests dbt-like (clés)
`order_id` unique et non NULL : déjà vérifié plus haut. Les indicateurs **ne font pas échouer** la construction :
ce sont des avertissements métier, pas des erreurs techniques.

## Pour ton journal (à rédiger avec tes mots)
1. Pourquoi `TRY_CAST` plutôt que `CAST` ?
2. Pourquoi on garde les lignes anormales avec un indicateur `flag_…` ?
3. Pourquoi Silver a le même nombre de lignes que Bronze ?